# Inter-Annotator Agreement (IAA) for Extracted Claim Terms

Compares the concept terms extracted by three annotators (**Tianyu**, **Mark**, **Gloria**)
from 100 patent claims, following `instructions.md`:

1. Normalize each annotator's raw text cell into a clean list of terms.
2. Find exact term matches per claim — the primary, trustworthy metric (zero ambiguity).
3. Compute strict pairwise Precision / Recall / F1 from those exact matches.
4. Summarize the strict scores — the headline number.
5. For terms left over after exact matching, compute how much of their vocabulary overlaps
   (Jaccard on tokens) — a descriptive statistic, not a match count or a claim about
   concept identity.
6. Summarize the average leftover-vocabulary overlap per pair.


In [1]:
import re
from itertools import combinations # For unique pairs of annotators

import pandas as pd

pd.set_option("display.max_colwidth", 200)

RAW_PATH = r"C:\Users\galassga\Documents\Projects\find-focal-terms\human_claim_annotations_100 - Tabellenblatt1.csv"
ANNOTATORS = ["Tianyu", "Mark", "Gloria"]
PAIRS = list(combinations(ANNOTATORS, 2))  # (Tianyu, Mark), (Tianyu, Gloria), (Mark, Gloria)

df_raw = pd.read_csv(RAW_PATH)
df_raw.head()


,patent_id,claim_number,claim_text,Tianyu,Mark,Gloria
0,7696175,33,"33. The method of claim 32 , wherein said soluble form of the co-stimulatory molecule is linked to another protein.",soluble form; co-stimulatory molecule; protein,co-stimulatory molecule; protein; soluble form,co-stimulatory molecule
1,7704241,5,"5. The absorbent article according to claim 4 , wherein said absorbent system comprises a material including a mixture of cellulosic fibers and superabsorbent material.",absorbent article; absorbent system; cellulosic fibers; superabsorbent material,absorbent system; cellulosic fibers; superabsorbent material,absorbent system; cellulosic fibers; superabsorbent material
2,7767672,54,"54. The compound of claim 52 , wherein said halogen of J is F.",halogen,halogen,"halogen F, halogen J"
3,7782077,31,"31. The method of claim 28 , further comprising filtering the first input pulse through a first input filter.",input pulse,input pulse; input filter,input pulse; input filter; filtering
4,7866377,9,"9. The method of claim 8 wherein the three-dimensional minimal skeleton heat exchanger forms a plate heat exchanger composed of multiple, thin slightly-separated plates that have large surface are...",heat exchanger; fluid flow passages; heat transfer,heat exchanger; flow passages; heat transfer; three-dimensional minimal skeleton; three-dimensional minimal skeleton heat exchanger,plate heat exchanger


## Step 1 — Normalize separators

Split on `;` always, and on `,` only when it's immediately followed by whitespace — a bare
`,` with no following space is part of a chemical name (e.g. `2,4-dichlorophenyl`,
`3,5-bis(trifluoromethyl)phenyl`) and must be left untouched. Each resulting term is
stripped and the list is de-duplicated case-insensitively, preserving order.


In [2]:
# Split on ';' always; split on ',' only when followed by whitespace (a tight comma,
# e.g. "2,4-dichlorophenyl", is part of a chemical name and must not be split on). 
SPLIT_RE = re.compile(r";\s*|,(?=\s)")


def normalize_terms(cell):
    """Split an annotator cell into a de-duplicated, order-preserving list of terms."""
    if not isinstance(cell, str) or not cell.strip():
        return []
    raw_terms = [t.strip() for t in SPLIT_RE.split(cell)]
    raw_terms = [t for t in raw_terms if t]
    seen, deduped = set(), []
    for term in raw_terms:
        key = term.lower()
        if key not in seen:
            seen.add(key)
            deduped.append(term)
    return deduped


# Sanity check: a purely-numeric "term" means the split rule broke a chemical name.
numeric_flags = []
for idx, row in df_raw.iterrows():
    for col in ANNOTATORS:
        for term in normalize_terms(row[col]):
            if term.replace(".", "", 1).isdigit():
                numeric_flags.append({
                    "row": idx, "patent_id": row["patent_id"], "annotator": col,
                    "flagged_term": term, "raw_cell": row[col],
                })

print(f"Numeric-term flags: {len(numeric_flags)} (0 expected on the current dataset)")
for flag in numeric_flags:
    print(flag)


Numeric-term flags: 0 (0 expected on the current dataset)


In [3]:
# Spot-check chemistry-heavy rows (rows containing at least one tight, no-space comma)
# to confirm those commas were preserved as part of the term rather than split on.
tight_comma_mask = df_raw[ANNOTATORS].apply(
    lambda col: col.str.contains(r",(?!\s)", regex=True, na=False)
).any(axis=1)

df_raw.loc[tight_comma_mask, ["patent_id", "claim_number"] + ANNOTATORS]


,patent_id,claim_number,Tianyu,Mark,Gloria
6,7915258,8,1-propylbutyl; cyclohexyl; 4-tert-butylcyclohexyl; 4-(trifluoromethyl)cyclohexyl; adamantan-1-yl; phenyl; 4-fluorophenyl; 2-methylphenyl; 4-methylphenyl; 4-isopropylphenyl; 4-butylphenyl; 4-tert-b...,"1-propylbutyl; cyclohexyl, 4-tert-butylcyclohexyl, 4-(trifluoromethyl)cyclohexyl; adamantan-1-yl; phenyl, 4-fluorophenyl, 2-methylphenyl, 4-methylphenyl, 4 isopropylphenyl, 4-butylphenyl, 4-tert-b...",compound of formula (I); 1-propylbutyl; cyclohexyl; 4-tert-butylcyclohexyl; 4-(trifluoromethyl)cyclohexyl; adamantan-1-yl; phenyl; 4-fluorophenyl; 2-methylphenyl; 4-methylphenyl; 4-isopropylphenyl...
36,8871972,10,"adapalene; adapalene methyl ester; hydrolyzing; adapalene salt; 3,3′-diadamantyl-4,4′-dimethoxybiphenyl; 3,3′-diadamantyl-4,4′-dimethoxybiphenyl impurity","method; adapalene; pharmaceutical use; adapalene methyl ester; hydrolyzing; adapalene salt; adapalene; 3,3′-diadamantyl-4,4′-dimethoxybiphenyl; reference marker; 3,3′-diadamantyl-4,4′-dimethoxybip...","adapalene; adapalene methyl ester; adapalene salt; 3,3′-diadamantyl-4,4′-dimethoxybiphenyl; reference marker; impurity"
65,9561309,20,"biocompatible polymer; poly(ester amides); polystyrene- polyisobutylene-polystyrene; block copolymers; polystyrene; polyisobutylene; polycaprolactone; poly(L-lactide); poly(D,L-lactide); poly(lact...",polymeric composition; biocompatible polymer; poly(ester amides); polystyrene- polyisobutylene-polystyrene; block copolymers (SIS); polystyrene; polyisobutylene; polycaprolactone (PCL); poly(L-lac...,polymeric composition; biocompatible polymer; poly(ester amides); polystyrene-polyisobutylene-polystyrene; block copolymers; polycaprolactone (PCL); polylactic acid (PLA); poly(glycolide); polydim...
87,10197567,9,"screening method; azoline compound; azoline compound library; azoline backbone; Cys; Ser; Thr; 2,3-diamino acid; Xaa 0; peptide; -(Xaa 0 ) m -; Xaa 0; amino acid; azoline ring; heterocyclase; hydr...","screening method; azoline compound; target substance; azoline compound library; azoline backbone; Cys; Ser; Thr; 2,3-diamino acid; analogs; Xaa 0; peptide; -(Xaa 0 ) m -; arbitrary amino acid; azo...",screening method; azoline compound; target substance; azoline compound library; azoline backbone; peptide; amino acid; azoline ring; heterocyclase; mRNA library; precursor peptides; cell-free tran...
97,10477209,3,image filtering; deblocked decoded image; pixel value; plurality of unit areas; image filtering device; input image; pixel value; subject pixel; classifying; offset class; subject pixel; offset va...,image filtering method; deblocked decoded image; pixel value; unit area; deblocked decoded image; image filtering device; offset value range; offset bit depth−K−1) −1); offset bit depth;\n SAO_DE...,image filtering method; deblocked decoded image; offset value; subject pixel; offset classes; offset bit depth


In [4]:
df_clean = df_raw.copy()
for col in ANNOTATORS:
    df_clean[col] = df_raw[col].apply(lambda cell: "; ".join(normalize_terms(cell)))

df_clean.head()


,patent_id,claim_number,claim_text,Tianyu,Mark,Gloria
0,7696175,33,"33. The method of claim 32 , wherein said soluble form of the co-stimulatory molecule is linked to another protein.",soluble form; co-stimulatory molecule; protein,co-stimulatory molecule; protein; soluble form,co-stimulatory molecule
1,7704241,5,"5. The absorbent article according to claim 4 , wherein said absorbent system comprises a material including a mixture of cellulosic fibers and superabsorbent material.",absorbent article; absorbent system; cellulosic fibers; superabsorbent material,absorbent system; cellulosic fibers; superabsorbent material,absorbent system; cellulosic fibers; superabsorbent material
2,7767672,54,"54. The compound of claim 52 , wherein said halogen of J is F.",halogen,halogen,halogen F; halogen J
3,7782077,31,"31. The method of claim 28 , further comprising filtering the first input pulse through a first input filter.",input pulse,input pulse; input filter,input pulse; input filter; filtering
4,7866377,9,"9. The method of claim 8 wherein the three-dimensional minimal skeleton heat exchanger forms a plate heat exchanger composed of multiple, thin slightly-separated plates that have large surface are...",heat exchanger; fluid flow passages; heat transfer,heat exchanger; flow passages; heat transfer; three-dimensional minimal skeleton; three-dimensional minimal skeleton heat exchanger,plate heat exchanger


## Step 2 — Exact matching (the primary, trustworthy metric)

Automating "is this a partial match" turned out to be unreliable no matter how the rule was
tuned (any-shared-token was too loose; substring-only missed things; same-head-word and a
noun-gate each fixed some cases and broke others). So **exact match is the primary metric**
from here on: two annotators either wrote the identical phrase (after lowercasing/stripping)
or they didn't — zero ambiguity, a legitimate IAA number on its own, no partial-credit
judgment call baked in.

Partial/related terms are no longer scored automatically. Step 5 generates candidate
"maybe-related" pairs with a deliberately loose heuristic and writes them out for manual
yes/no review; the *lenient* score (Step 6) is built from those reviewed judgments, not from
the heuristic itself.

Matching is always done **within a single claim** — the per-row loop in Step 3 calls
`find_exact_matches` once per (claim, annotator pair) on that row's own term lists only.


In [5]:
def find_exact_matches(list_a, list_b):
    """Match two term lists FROM THE SAME CLAIM on exact (lowercased/stripped) equality
    only - the primary, unambiguous metric. Each term is used at most once.

    Returns (exact_matches, unmatched_a, unmatched_b), where exact_matches is a list of
    (term_a, term_b) tuples and unmatched_a/b are the leftover terms on each side (inputs
    to the Step 5 candidate generator).
    """
    a_items, b_items = list(enumerate(list_a)), list(enumerate(list_b))
    matched_a, matched_b = set(), set()
    exact_matches = []

    for i, a_term in a_items:
        a_norm = a_term.strip().lower()
        for j, b_term in b_items:
            if j in matched_b:
                continue
            if b_term.strip().lower() == a_norm:
                exact_matches.append((a_term, b_term))
                matched_a.add(i)
                matched_b.add(j)
                break

    unmatched_a = [t for i, t in a_items if i not in matched_a]
    unmatched_b = [t for j, t in b_items if j not in matched_b]
    return exact_matches, unmatched_a, unmatched_b


# Quick demo: claim 0 (Tianyu vs Mark) and the halogen claim (patent 7767672, claim 54)
demo_a = normalize_terms(df_raw.loc[0, "Tianyu"])
demo_b = normalize_terms(df_raw.loc[0, "Mark"])
print("Claim 0, Tianyu vs Mark:", find_exact_matches(demo_a, demo_b))

halogen_row = df_raw[df_raw["patent_id"] == 7767672].iloc[0]
h_tianyu = normalize_terms(halogen_row["Tianyu"])
h_gloria = normalize_terms(halogen_row["Gloria"])
print("Halogen claim, Tianyu vs Gloria:", find_exact_matches(h_tianyu, h_gloria))


Claim 0, Tianyu vs Mark: ([('soluble form', 'soluble form'), ('co-stimulatory molecule', 'co-stimulatory molecule'), ('protein', 'protein')], [], [])
Halogen claim, Tianyu vs Gloria: ([], ['halogen'], ['halogen F', 'halogen J'])


## Step 3 — Strict Precision / Recall / F1 (primary metric)

For each annotator pair (Tianyu-Mark, Tianyu-Gloria, Mark-Gloria) and each claim:

- `matches` = exact matches only (Step 2)
- `Precision = matches / len(terms_B)`, `Recall = matches / len(terms_A)`,
  `F1 = 2*P*R/(P+R)`

Since there's no ground truth, both **reference directions** are computed per pair and kept
in `df_detail` for drill-down. F1 is identical in both directions since it's symmetric.

This is the trustworthy, headline number — zero ambiguity, no partial-credit judgment call.
Steps 5-6 add a *lenient* score on top once the partial-match candidates have been manually
reviewed.


In [6]:
def safe_div(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def f1_from_pr(precision, recall):
    return safe_div(2 * precision * recall, precision + recall)


detail_rows = []
for idx, row in df_raw.iterrows():
    terms = {ann: normalize_terms(row[ann]) for ann in ANNOTATORS}
    for ann_a, ann_b in PAIRS:
        list_a, list_b = terms[ann_a], terms[ann_b]
        exact, unmatched_a, unmatched_b = find_exact_matches(list_a, list_b)
        n_exact = len(exact)
        len_a, len_b = len(list_a), len(list_b)

        for ref, num_ref, num_other in [("a", len_a, len_b), ("b", len_b, len_a)]:
            precision_strict = safe_div(n_exact, num_other)
            recall_strict = safe_div(n_exact, num_ref)

            detail_rows.append({
                "patent_id": row["patent_id"],
                "claim_number": row["claim_number"],
                "annotator_a": ann_a,
                "annotator_b": ann_b,
                "reference": ann_a if ref == "a" else ann_b,
                "n_terms_a": len_a,
                "n_terms_b": len_b,
                "exact_matches": exact,
                "unmatched_to_a": unmatched_a,
                "unmatched_to_b": unmatched_b,
                "n_exact": n_exact,
                "precision_strict": precision_strict,
                "recall_strict": recall_strict,
                "f1_strict": f1_from_pr(precision_strict, recall_strict),
            })

df_detail = pd.DataFrame(detail_rows)
df_detail.head(10)


,patent_id,claim_number,annotator_a,annotator_b,reference,n_terms_a,n_terms_b,exact_matches,unmatched_to_a,unmatched_to_b,n_exact,precision_strict,recall_strict,f1_strict
0,7696175,33,Tianyu,Mark,Tianyu,3,3,"[(soluble form, soluble form), (co-stimulatory molecule, co-stimulatory molecule), (protein, protein)]",[],[],3,1.000000,1.000000,1.000000
1,7696175,33,Tianyu,Mark,Mark,3,3,"[(soluble form, soluble form), (co-stimulatory molecule, co-stimulatory molecule), (protein, protein)]",[],[],3,1.000000,1.000000,1.000000
2,7696175,33,Tianyu,Gloria,Tianyu,3,1,"[(co-stimulatory molecule, co-stimulatory molecule)]","[soluble form, protein]",[],1,1.000000,0.333333,0.500000
3,7696175,33,Tianyu,Gloria,Gloria,3,1,"[(co-stimulatory molecule, co-stimulatory molecule)]","[soluble form, protein]",[],1,0.333333,1.000000,0.500000
4,7696175,33,Mark,Gloria,Mark,3,1,"[(co-stimulatory molecule, co-stimulatory molecule)]","[protein, soluble form]",[],1,1.000000,0.333333,0.500000
5,7696175,33,Mark,Gloria,Gloria,3,1,"[(co-stimulatory molecule, co-stimulatory molecule)]","[protein, soluble form]",[],1,0.333333,1.000000,0.500000
6,7704241,5,Tianyu,Mark,Tianyu,4,3,"[(absorbent system, absorbent system), (cellulosic fibers, cellulosic fibers), (superabsorbent material, superabsorbent material)]",[absorbent article],[],3,1.000000,0.750000,0.857143
7,7704241,5,Tianyu,Mark,Mark,4,3,"[(absorbent system, absorbent system), (cellulosic fibers, cellulosic fibers), (superabsorbent material, superabsorbent material)]",[absorbent article],[],3,0.750000,1.000000,0.857143
8,7704241,5,Tianyu,Gloria,Tianyu,4,3,"[(absorbent system, absorbent system), (cellulosic fibers, cellulosic fibers), (superabsorbent material, superabsorbent material)]",[absorbent article],[],3,1.000000,0.750000,0.857143
9,7704241,5,Tianyu,Gloria,Gloria,4,3,"[(absorbent system, absorbent system), (cellulosic fibers, cellulosic fibers), (superabsorbent material, superabsorbent material)]",[absorbent article],[],3,0.750000,1.000000,0.857143


## Step 4 — Strict summary (primary, headline metric)

`df_summary_strict` reports precision, recall, and F1 from exact matches only, averaged
across all 100 claims, per annotator pair *and* reference direction, plus each annotator's
average across their two pairings. **This is the number to lead with** — zero ambiguity, no
partial-credit judgment call. Steps 5-7 build a lenient number on top of it once the
partial-match candidates have been manually reviewed.


In [7]:
strict_cols = ["precision_strict", "recall_strict", "f1_strict"]

# Pair-level averages, kept per reference direction since precision/recall depend on
# which annotator's terms are treated as the reference (F1 is symmetric and repeats).
pair_summary_strict = df_detail.groupby(["annotator_a", "annotator_b", "reference"])[strict_cols].mean().reset_index()
pair_summary_strict.insert(0, "scope", "pair")
pair_summary_strict["n_claims"] = len(df_raw)

# Per-annotator averages across their two pairings, using that annotator as the reference.
annotator_rows = []
for annotator in ANNOTATORS:
    relevant = pair_summary_strict[pair_summary_strict["reference"] == annotator]
    means = relevant[strict_cols].mean()
    annotator_rows.append({
        "scope": "annotator",
        "annotator_a": annotator,
        "annotator_b": None,
        "reference": annotator,
        "n_claims": len(df_raw),
        **{col: means[col] for col in strict_cols},
    })

df_summary_strict = pd.concat([pair_summary_strict, pd.DataFrame(annotator_rows)], ignore_index=True)
df_summary_strict


,scope,annotator_a,annotator_b,reference,precision_strict,recall_strict,f1_strict,n_claims
0,pair,Mark,Gloria,Gloria,0.574912,0.732594,0.626968,100
1,pair,Mark,Gloria,Mark,0.732594,0.574912,0.626968,100
2,pair,Tianyu,Gloria,Gloria,0.642769,0.715410,0.656981,100
3,pair,Tianyu,Gloria,Tianyu,0.715410,0.642769,0.656981,100
4,pair,Tianyu,Mark,Mark,0.740865,0.651453,0.681785,100
5,pair,Tianyu,Mark,Tianyu,0.651453,0.740865,0.681785,100
6,annotator,Tianyu,None,Tianyu,0.683431,0.691817,0.669383,100
7,annotator,Mark,None,Mark,0.736730,0.613183,0.654376,100
8,annotator,Gloria,None,Gloria,0.608841,0.724002,0.641974,100


## Step 5 — Leftover vocabulary overlap (descriptive, not a match count)

For terms left over after exact matching (Step 2), automatically deciding "is this the same
concept" turned out to be unreliable no matter how the rule was tuned. Rather than score
leftover terms as matches at all, this reports something purely descriptive: what fraction
of the *combined vocabulary* of A's leftover terms and B's leftover terms is shared — a
Jaccard index over tokens (alphanumeric words, lowercased, stopwords and single-character
tokens dropped), computed once per claim per annotator pair.

This is **not** a claim about concept identity. It says nothing about whether any specific
leftover term pair refers to the same thing — two annotators could share 80% of their
leftover vocabulary while describing different sub-features of the claim, or share 0% while
still meaning something related in different words. It's reported as "leftover terms shared
X% of their vocabulary," not as a match count, and stays out of any P/R/F1 computation.


In [8]:
VOCAB_STOPWORDS = {"a", "an", "the", "of", "said", "and", "or", "is", "in", "to", "for", "on", "with", "at"}
VOCAB_TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(term):
    tokens = VOCAB_TOKEN_RE.findall(term.lower())
    return {t for t in tokens if t not in VOCAB_STOPWORDS and len(t) > 1}


def vocab(terms):
    vocab_tokens = set()
    for term in terms:
        vocab_tokens |= tokenize(term)
    return vocab_tokens


def vocab_jaccard(vocab_a, vocab_b):
    union = vocab_a | vocab_b
    if not union:
        return None  # nothing left over on either side - not applicable
    return len(vocab_a & vocab_b) / len(union)


overlap_rows = []
for idx, row in df_raw.iterrows():
    terms = {ann: normalize_terms(row[ann]) for ann in ANNOTATORS}
    for ann_a, ann_b in PAIRS:
        _, unmatched_a, unmatched_b = find_exact_matches(terms[ann_a], terms[ann_b])
        vocab_a, vocab_b = vocab(unmatched_a), vocab(unmatched_b)
        overlap_rows.append({
            "patent_id": row["patent_id"],
            "claim_number": row["claim_number"],
            "annotator_a": ann_a,
            "annotator_b": ann_b,
            "n_leftover_a": len(unmatched_a),
            "n_leftover_b": len(unmatched_b),
            "leftover_terms_a": unmatched_a,
            "leftover_terms_b": unmatched_b,
            "leftover_vocab_jaccard": vocab_jaccard(vocab_a, vocab_b),
        })

df_leftover_vocab = pd.DataFrame(overlap_rows)
df_leftover_vocab.head(10)


,patent_id,claim_number,annotator_a,annotator_b,n_leftover_a,n_leftover_b,leftover_terms_a,leftover_terms_b,leftover_vocab_jaccard
0,7696175,33,Tianyu,Mark,0,0,[],[],NaN
1,7696175,33,Tianyu,Gloria,2,0,"[soluble form, protein]",[],0.0
2,7696175,33,Mark,Gloria,2,0,"[protein, soluble form]",[],0.0
3,7704241,5,Tianyu,Mark,1,0,[absorbent article],[],0.0
4,7704241,5,Tianyu,Gloria,1,0,[absorbent article],[],0.0
5,7704241,5,Mark,Gloria,0,0,[],[],NaN
6,7767672,54,Tianyu,Mark,0,0,[],[],NaN
7,7767672,54,Tianyu,Gloria,1,2,[halogen],"[halogen F, halogen J]",1.0
8,7767672,54,Mark,Gloria,1,2,[halogen],"[halogen F, halogen J]",1.0
9,7782077,31,Tianyu,Mark,0,1,[],[input filter],0.0


## Step 6 — Average leftover-vocabulary overlap per pair

Averages `leftover_vocab_jaccard` across claims, per annotator pair, skipping claims where
it's not applicable (both sides had nothing left over after exact matching — i.e. every
term already matched exactly, so there's no leftover vocabulary to compare). Reported as
"leftover terms shared X% of their vocabulary," alongside `df_summary_strict` (Step 4) —
the two are independent, complementary readings, not two versions of the same score.


In [9]:
applicable = df_leftover_vocab.dropna(subset=["leftover_vocab_jaccard"])

df_vocab_summary = (
    applicable.groupby(["annotator_a", "annotator_b"])["leftover_vocab_jaccard"]
    .agg(mean_jaccard="mean", n_claims_with_leftovers="count")
    .reset_index()
)
df_vocab_summary["n_claims_total"] = len(df_raw)
df_vocab_summary["pct_shared_vocab"] = (df_vocab_summary["mean_jaccard"] * 100).round(1)

for _, r in df_vocab_summary.iterrows():
    print(
        f"{r['annotator_a']}-{r['annotator_b']}: leftover terms shared "
        f"{r['pct_shared_vocab']}% of their vocabulary on average "
        f"(across {r['n_claims_with_leftovers']} of {r['n_claims_total']} claims "
        f"with leftover terms on both sides)."
    )

df_vocab_summary


Mark-Gloria: leftover terms shared 28.6% of their vocabulary on average (across 92 of 100 claims with leftover terms on both sides).
Tianyu-Gloria: leftover terms shared 30.9% of their vocabulary on average (across 81 of 100 claims with leftover terms on both sides).
Tianyu-Mark: leftover terms shared 32.0% of their vocabulary on average (across 87 of 100 claims with leftover terms on both sides).


,annotator_a,annotator_b,mean_jaccard,n_claims_with_leftovers,n_claims_total,pct_shared_vocab
0,Mark,Gloria,0.285731,92,100,28.6
1,Tianyu,Gloria,0.309272,81,100,30.9
2,Tianyu,Mark,0.320438,87,100,32.0
